In [16]:
import pandas as pd
import numpy as np
import seaborn as sns
import pyreadstat as pyr
import statsmodels.formula.api as smf



In [2]:
# 导入数据，描述性统计，清洗
df, meta = pyr.read_dta(r"C:\Users\Friday\OneDrive\Desktop\statistics\计量经济学\因果推断(数据集）\第三讲演示数据和do文件\DD_wide.dta")
null_list = {}
for column in df.columns:
    null_num = df[column].isnull().sum()
    null_list[column] = null_num
print(null_list)
df.describe()

{'id': np.int64(0), 'fisrev2': np.int64(0), 'exp2': np.int64(0), 'fisrev1': np.int64(0), 'exp1': np.int64(0), 'treat': np.int64(0)}


,id,fisrev2,exp2,fisrev1,exp1,treat
count,1062.000000,1062.000000,1062.000000,1062.000000,1062.000000,1062.000000
mean,663.525424,11615.159066,1783.289018,9708.927236,1226.910088,0.191149
std,438.775254,9750.416642,730.791971,9110.900399,632.826619,0.393391
min,2.000000,541.804993,639.612976,502.696014,345.682007,0.000000
25%,296.250000,4164.682495,1261.281769,3296.666992,834.910522,0.000000
50%,591.500000,8909.780762,1604.451477,6907.113525,1086.765503,0.000000
75%,988.750000,15843.175293,2139.964722,12694.218750,1452.867462,0.000000
max,1529.000000,49841.914062,5527.094238,49451.789062,8084.583008,1.000000


In [3]:
df.head(10)

,id,fisrev2,exp2,fisrev1,exp1,treat
0,2,15177.071289,3013.943115,8158.963867,970.947021,0.0
1,3,23095.658203,3192.278076,17955.931641,2117.831055,0.0
2,4,13042.378906,2263.341064,7595.400879,1308.708008,0.0
3,5,25833.671875,1391.187012,17859.964844,1052.800049,0.0
4,6,26527.490234,1509.189941,19080.234375,1045.272949,0.0
5,8,15304.833008,1788.668945,8920.103516,1224.032959,0.0
6,9,5753.729980,1906.671997,4144.399902,1302.123047,0.0
7,10,7247.836914,1990.072021,3520.623047,1558.972046,0.0
8,12,4082.176025,2360.050049,3171.572021,2183.689941,0.0
9,13,20611.396484,1612.995972,9963.495117,1210.860962,0.0


In [18]:
treat_counts = df["treat"].value_counts()
print(treat_counts)

treat
0.0    859
1.0    203
Name: count, dtype: int64


In [6]:
# 原数据集为宽面版，转化为长面板
df_long = pd.wide_to_long(
    df,
    stubnames=["fisrev","exp"],
    i=["id","treat"],
    j="year"    
).reset_index()

df_long.head()





,id,treat,year,fisrev,exp
0,2,0.0,2,15177.071289,3013.943115
1,2,0.0,1,8158.963867,970.947021
2,3,0.0,2,23095.658203,3192.278076
3,3,0.0,1,17955.931641,2117.831055
4,4,0.0,2,13042.378906,2263.341064


In [29]:
df_long.describe()
df_long["treat"] = df_long["treat"].astype("float").astype("int")

In [30]:
# 不控制人均财政收入和控制人均财政收入前提下回归
model_1 = smf.ols("exp~treat+year",data=df_long)
result_1 = model_1.fit()
print(result_1.summary())


                            OLS Regression Results                            
Dep. Variable:                    exp   R-squared:                       0.186
Model:                            OLS   Adj. R-squared:                  0.185
Method:                 Least Squares   F-statistic:                     241.6
Date:                Sun, 14 Jun 2026   Prob (F-statistic):           2.93e-95
Time:                        16:10:27   Log-Likelihood:                -16822.
No. Observations:                2124   AIC:                         3.365e+04
Df Residuals:                    2121   BIC:                         3.367e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    595.8608     46.250     12.883      0.0

In [31]:
model_2 = smf.ols("exp~treat+year+fisrev",data=df_long)
result_2 = model_2.fit()
print(result_2.summary())

                            OLS Regression Results                            
Dep. Variable:                    exp   R-squared:                       0.193
Model:                            OLS   Adj. R-squared:                  0.192
Method:                 Least Squares   F-statistic:                     168.7
Date:                Sun, 14 Jun 2026   Prob (F-statistic):           4.38e-98
Time:                        16:10:34   Log-Likelihood:                -16812.
No. Observations:                2124   AIC:                         3.363e+04
Df Residuals:                    2120   BIC:                         3.366e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    648.2313     47.609     13.616      0.0

In [ ]:
''' 线性回归估计模式下的DID '''
# 1、不控制fisrev的双重差分
df_long["year_treat"] = df_long["year"]*df_long["treat"]
did_1 = smf.ols("exp~treat+year+year_treat",data=df_long)
did_1 = did_1.fit()
print(did_1.summary())


                            OLS Regression Results                            
Dep. Variable:                    exp   R-squared:                       0.187
Model:                            OLS   Adj. R-squared:                  0.186
Method:                 Least Squares   F-statistic:                     162.7
Date:                Sun, 14 Jun 2026   Prob (F-statistic):           6.74e-95
Time:                        16:10:41   Log-Likelihood:                -16820.
No. Observations:                2124   AIC:                         3.365e+04
Df Residuals:                    2120   BIC:                         3.367e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    638.4225     50.792     12.569      0.0

In [33]:
# 2、只控制fisrev
did_2 = smf.ols("exp~treat+year+year_treat+fisrev",data=df_long)
did_2 = did_2.fit()
print(did_2.summary())


                            OLS Regression Results                            
Dep. Variable:                    exp   R-squared:                       0.194
Model:                            OLS   Adj. R-squared:                  0.193
Method:                 Least Squares   F-statistic:                     127.6
Date:                Sun, 14 Jun 2026   Prob (F-statistic):           9.74e-98
Time:                        16:10:59   Log-Likelihood:                -16811.
No. Observations:                2124   AIC:                         3.363e+04
Df Residuals:                    2119   BIC:                         3.366e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    688.1344     51.888     13.262      0.0

In [34]:
# 3、除了fisrev外，还控制fisrev和treat、year的交互项
df_long["year_fisrev"] = df_long["year"]*df_long["fisrev"]
df_long["treat_fisrev"] = df_long["treat"]*df_long["fisrev"]
did_3 = smf.ols("exp~treat+year+year_treat+fisrev+year_fisrev+treat_fisrev",data=df_long)
did_3 = did_3.fit()
print(did_3.summary())

                            OLS Regression Results                            
Dep. Variable:                    exp   R-squared:                       0.195
Model:                            OLS   Adj. R-squared:                  0.193
Method:                 Least Squares   F-statistic:                     85.59
Date:                Sun, 14 Jun 2026   Prob (F-statistic):           3.14e-96
Time:                        16:11:31   Log-Likelihood:                -16809.
No. Observations:                2124   AIC:                         3.363e+04
Df Residuals:                    2117   BIC:                         3.367e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept      769.7098     71.349     10.788   

In [37]:
''' 固定效应估计模式的实现，使用宽面板df '''
# 1、不控制人均财政收入
df["d_exp"] = df["exp2"]-df["exp1"]
df["d_fisrev"] = df["fisrev2"]-df["fisrev1"]
did_4 = smf.ols("d_exp~treat",data=df)
did_4 = did_4.fit()
print(did_4.summary())


                            OLS Regression Results                            
Dep. Variable:                  d_exp   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     15.04
Date:                Sun, 14 Jun 2026   Prob (F-statistic):           0.000112
Time:                        16:21:12   Log-Likelihood:                -8085.3
No. Observations:                1062   AIC:                         1.617e+04
Df Residuals:                    1060   BIC:                         1.618e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    528.0045     16.733     31.554      0.0

In [38]:
# 2. 控制人均财政收入，取变化值df["d_fisrev"]
did_5 = smf.ols("d_exp~treat+d_fisrev",data=df)
did_5 = did_5.fit()
print(did_5.summary())

                            OLS Regression Results                            
Dep. Variable:                  d_exp   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     7.533
Date:                Sun, 14 Jun 2026   Prob (F-statistic):           0.000564
Time:                        16:22:37   Log-Likelihood:                -8085.3
No. Observations:                1062   AIC:                         1.618e+04
Df Residuals:                    1059   BIC:                         1.619e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    527.4811     16.960     31.102      0.0